In [ ]:
!pip install -q --upgrade transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.9 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login()

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving truthfulqa_cleaned.csv to truthfulqa_cleaned.csv


In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer loaded!")

Loading tokenizer...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded!


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

print("Loading model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config
)

model.eval()

print("Model loaded!")

Loading model...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded!


In [ ]:
DATA_PATH = "truthfulqa_cleaned.csv"
OUTPUT_PATH = "llama3_output.csv"

print("Paths set!")

Paths set!


In [ ]:
import pandas as pd
from tqdm import tqdm
import torch

# Load dataset
df = pd.read_csv(DATA_PATH)

if "question" in df.columns:
    questions = df["question"].tolist()
elif "Question" in df.columns:
    questions = df["Question"].tolist()
else:
    raise ValueError("No question column found!")

questions = questions[:800]


def generate_answer(question):
    messages = [
        {"role": "user", "content": question}
    ]

    # 🔥 Get properly structured inputs
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    )

    # 🔥 Extract tensors correctly
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=75,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response.strip()


results = []

for q in tqdm(questions):
    try:
        ans = generate_answer(q)
    except Exception as e:
        print(f"\n❌ Error for question: {q}")
        print(e)
        ans = "ERROR"

    results.append({
        "question": q,
        "model_output": ans
    })


pd.DataFrame(results).to_csv(OUTPUT_PATH, index=False)

print("Saved to", OUTPUT_PATH)

100%|██████████| 800/800 [1:28:54<00:00,  6.67s/it]

Saved to llama3_output.csv


In [ ]:
from google.colab import files
files.download("llama3_output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>